# 📘 Annotated edition · 03_clean_reconcile

A copy of `notebooks/03_clean_reconcile.ipynb`. Original cells and saved outputs are unchanged. New **📘 Explanation** and **✅ Answer** cells are added.

**The point of this notebook:** every cleaning step is a **decision** with a reason, a row count and an effect on the headline. After every step *input = kept + rejected*, and after every join the row count is checked.

**Key results from the saved run:**
| Step | Before → after |
|---|---|
| drop superseded nova records (identity rule) | 564,879 → **563,380** bookings, 1,307,260 → **1,303,719** tests |
| collections double scans | 552,492 → **549,598** rows (2,894 removed) |
| bookings LEFT JOIN collections | 563,380 → 563,380 (no fan-out). 545,111 matched, **18,269** unmatched |
| clock-skew timestamps blanked | 936 |
| patients de-duplicated | 309,217 ids → **303,153** people |
| invoices / payments | 470,776 invoices (₹737.0 M), 473,507 payment rows (455,469 captured) |

⚠️ **A significant problem this annotated copy points out (cells 5 and 12):** the phlebotomy app records a rescheduled nova visit's collection against the **original** booking id. Dropping the originals before the collections join makes **all 1,500 rescheduled visits look "cancelled_or_missing"**, although 1,449 of them have a collection record (1,239 collected, 210 no-show). A fix is suggested below cell 12.

> ℹ️ **One line changed in this annotated copy:** `ROOT = ... if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else ...`. The original only recognised a folder called `notebooks`. Run from `notebook_new/`, it would have looked for `notebook_new/data/raw` and failed. Nothing else in the code was changed.

# 03 · Clean, decide, reconcile

**Produces:** `data/clean/bookings_clean.csv.gz`, `data/clean/collections_clean.csv.gz`, `data/clean/patients_clean.csv.gz`, `data/clean/invoices_clean.csv.gz`, `data/clean/payments_clean.csv.gz`, and rows in the decisions log for every rule applied.

**Needs:** notebook 02 run.

Every cleaning act here is a decision with a reason, a row count and an effect on the headline. Input equals clean plus rejected after every step. Joins are checked before and after on row count, and a join that grows is explained or reverted.

#### 📘 Explanation: header

Five clean tables come out of this notebook: bookings (now with collection status and `person_id`), collections, patients (one row per person), invoices (amount as a number) and payments (rupees, IST timestamps, normalised invoice id).

"A join that grows is explained or reverted": if a LEFT join returns more rows than its left table, the right table has repeated keys. Either you understand why (and it's intended) or you undo it.

## CONFIG

#### 📘 Explanation: the CONFIG cell, line by line

This cell is the same in notebooks 01–06. Explained once in full:

| Code | What it does and why |
|---|---|
| `import sys, pathlib, time` | `sys` to edit the import path, `pathlib` for file paths that work on Windows and Linux, `time` to time steps. |
| `ROOT = pathlib.Path.cwd().parent if ... == "notebooks" else pathlib.Path.cwd()` | Finds the **project root**. Jupyter usually starts inside the notebook's folder, so the root is one level up. When run from the root itself (for example a script), `cwd()` is already the root. Every path below is built from `ROOT`, so the notebook works on any machine. |
| `sys.path.insert(0, str(ROOT / "src"))` | Puts `src/` at the front of Python's import search path, so `from kalpa import ...` finds the project's own package without `pip install`. |
| `import numpy as np` / `import pandas as pd` | The two core libraries: arrays and fast maths / tables. |
| `from kalpa import loaders, models, checks, logs` | The shared project package: **loaders** read the raw files, **models** are pydantic "contracts" for a valid record, **checks** print PASS/FAIL, **logs** append rows to the markdown logs in `docs/`. |
| `pd.set_option("display.max_columns", 40)` / `("display.width", 160)` | Print wide tables in full instead of truncating them with `...`. |
| `CONFIG = {...}` | All run-specific settings in one place (see notebook 00): brief, group, `raw` and `clean` folders as `pathlib` paths, the two half-years being compared, the seed, and `workers` (`None` means use every CPU core in the process pool). |
| `CONFIG["clean"].mkdir(parents=True, exist_ok=True)` | Creates `data/clean/` if it doesn't exist, and doesn't fail if it does. A fresh clone can run straight away. |
| `rng = np.random.default_rng(CONFIG["seed"])` | A seeded random generator **object**. Anything random (the shuffle test in 05) uses `rng`, so results are reproducible. |
| `print(pd.__version__, np.__version__)` | Prints library versions (**2.3.3** and **2.3.5** in the saved run). If a re-run elsewhere gives different numbers, a version difference is the first suspect. pandas 3.x, for example, changed the default string dtype. |

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "<LETTER>",
    "group": "<N>",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

2.3.3 2.3.5


#### 📘 Explanation: load the booking table and define the headline

- `pd.read_csv(... "bookings.csv.gz", parse_dates=[...], dtype={...: "string"})`: CSV has no types, so dates are re-parsed and id columns are forced to `string`. Otherwise pandas might read an all-empty column (`rescheduled_from` is empty for every legacy row) as `float64` NaN.
- `HEADLINE = lambda df: f"{len(df):,} bookings, {int(df['n_tests'].sum()):,} tests"`: a tiny function that prints the headline the same way after every step. That builds the **bridge** from raw to clean.
- `n0 = len(bookings)` stores the starting count for the reconciliation.

In [2]:
bookings = pd.read_csv(CONFIG["clean"] / "bookings.csv.gz", parse_dates=["created_at", "slot_at"], dtype={"booking_id": "string", "patient_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
HEADLINE = lambda df: f"{len(df):,} bookings, {int(df['n_tests'].sum()):,} tests"
n0 = len(bookings)
print(HEADLINE(bookings))

564,879 bookings, 1,307,260 tests


#### 📘 Reading the output
**564,879 bookings, 1,307,260 tests**: exactly what notebook 02 accepted. (Tests = sum of `n_tests`. That equals 1,315,183 booking-test rows minus the rows belonging to the 3,107 rejected bookings.)

## 1 · DO: the identity rule for a booking

Nova records a rescheduled visit as a new booking that points at the old one, and marks the old one `rescheduled`. Legacy overwrote the slot in place. If the question is "how many visits happened", a rescheduled pair is one visit. Write the rule, apply it, log it.

#### 📘 Explanation: the identity rule

Two systems record the same event differently:
- **Legacy:** patient reschedules, and the vendor **overwrites** `slot_at` on the same booking. One id, one row.
- **Nova:** patient reschedules, so nova **creates a new booking** (`rescheduled_from = <old id>`) and marks the old one `status_at_export = "rescheduled"`. Two ids.

If the question is "how many visits happened", a nova pair must count as **one**, like legacy. Rule: **drop the superseded original, keep the new record** (which has the slot that actually happened).

#### 📘 Explanation: applying the rule, and the `pd.NA` trap

- `bookings["status_at_export"].eq("rescheduled")`: for nova rows this is True/False. **For legacy rows `status_at_export` is `<NA>`, and `NA == "rescheduled"` gives `<NA>`, not False.** With the nullable `string` dtype, `.eq()` returns a nullable `boolean` series with NAs in it.
- `.fillna(False)` turns those NAs into False ("legacy rows are not superseded"). `.astype(bool)` makes it a plain boolean array.
- Without `fillna`: `~is_superseded` would be `~NA = NA`, and filtering a DataFrame with a mask containing NA either drops those rows or raises, depending on version. **All ~538,000 legacy bookings could silently disappear.** That's the trap the comment describes.
- `kept = bookings[~is_superseded].copy()`. `.copy()` makes an independent frame, which avoids `SettingWithCopyWarning` when we add columns later.
- `checks.reconciles("drop superseded nova records", n0, len(kept), len(rejected))` checks in = kept + rejected.
- The `logs.decision(...)` call is commented out. Uncomment it, put your name in, and run it **once** to record the rule (numbers: 1,499 rows, 564,879 → 563,380).

In [3]:
# status_at_export is NA for every legacy row. A nullable boolean treats NA as False only if you say so; left alone,
# ~NA is NA and the legacy rows vanish from the filter. The trap is worth seeing once, so it is stated here.
is_superseded = bookings["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
kept = bookings[~is_superseded].copy()
rejected = bookings[is_superseded]
checks.reconciles("drop superseded nova records", n0, len(kept), len(rejected))
print("after:", HEADLINE(kept))
# logs.decision("bookings", "nova writes a new record on reschedule and marks the old one", "drop the superseded original",
#               "one visit, one booking; legacy overwrote in place so this makes the systems comparable",
#               rows_affected=len(rejected), effect_on_headline=f"{n0:,} -> {len(kept):,} bookings", owner="<name>", n=2)
bookings = kept

PASS  drop superseded nova records: input = kept + rejected   564,879 = 563,380 + 1,499
after: 563,380 bookings, 1,303,719 tests


#### 📘 Reading the output

- **PASS: 564,879 = 563,380 + 1,499.**
- **After: 563,380 bookings, 1,303,719 tests**, so 3,541 tests removed (about 2.36 per dropped booking).
- All 1,499 dropped records are nova, so all are in **Chennai and Jaipur, April–June 2026**. This rule lowers **H1 2026 only**: on the H1 comparison it moves growth from +5.84% to **+5.02%** (about −0.8 points). It's the rule that moves the headline most.

⚠️ **Side effect, important:** the phlebotomy app wrote collections against the **original** booking id. After this drop, the collection row for a rescheduled visit no longer has a booking to join to, and the new nova record has no collection row. See the fix after cell 12.

> **Interview angle.** What is `pd.NA`, and what happens to `~series.eq('x')` when the series holds NA?
>
> *What a good answer covers:* pd.NA is the missing marker for nullable dtypes and propagates through comparisons, so eq gives NA, the negation stays NA, and a boolean filter drops those rows silently. fillna(False) before negating, or compare with notna() explicitly.

#### ✅ Answer: what is `pd.NA`, and what happens to `~series.eq('x')` when the series holds NA?

**`pd.NA`** is pandas' single "missing value" marker for the **nullable dtypes** (`string`, `Int64`/`Int32`, `boolean`, `Float64`). Unlike `np.nan` (a float), it can sit in an integer, string or boolean column without converting the column's type. It follows **three-valued logic**: an unknown compared with anything is unknown.

**What happens step by step**, for a nullable string column `["rescheduled", "booked", <NA>]`:
1. `series.eq("rescheduled")` gives `[True, False, <NA>]`. The comparison with a missing value is *unknown*, not False.
2. `~(...)` gives `[False, True, <NA>]`. "Not unknown" is still unknown.
3. `df[mask]`: depending on pandas version, the `<NA>` rows are **dropped silently**, or it raises `ValueError: cannot mask with non-boolean array containing NA / NaN values`.

In this notebook every legacy row has `<NA>` in `status_at_export`, so the silent version would remove **about 538,000 legacy bookings** while looking like a harmless filter that only touched nova.

**The fix:** decide what NA means *before* negating: `series.eq("x").fillna(False)` (missing means "not x"), or be explicit: `series.notna() & series.eq("x")`. (Contrast with `np.nan`: `np.nan == "x"` is plain `False`, so object-dtype columns don't show the trap, which is exactly why it surprises people when they switch to nullable dtypes.)

Related: `pd.NA | True` is `True` and `pd.NA & False` is `False` (Kleene logic: the result is known whatever the missing value is), but `pd.NA & True` is `NA`. And `if pd.NA:` raises `TypeError` because its truth value is ambiguous.

> **Interview angle.** Two systems record the same business event in different shapes. What is the general method for making them comparable, and what is the trap?
>
> *What a good answer covers:* Trace one real event through both systems by hand and write down how each represents it; then write a rule that maps both to one definition of the event; then apply it and check the count before and after. The trap is comparing totals without the rule and attributing the difference to the business.

#### ✅ Answer: two systems record the same event in different shapes. What is the general method for making them comparable, and what is the trap?

**The method:**
1. **Trace one real event by hand through both systems.** Pick a rescheduled visit, find it in legacy (one row, slot overwritten) and in nova (two records linked by `rescheduled_from`, the old one marked `rescheduled`). Also find its collection row (here: under the *original* id).
2. **Write down how each system represents the event**: which fields, how many records, which id survives, what the downstream systems (collections, results, billing) point at.
3. **Write one business definition of the event** that doesn't depend on either system: "a booking is one visit by one patient".
4. **Write the mapping rule** from each system to that definition: drop nova's superseded original, carry its collection over to the new record, and leave legacy unchanged.
5. **Apply it and reconcile**: count before and after (in = kept + rejected), check joins don't fan out, and recompute the headline.
6. **Log the rule** with its alternatives and its effect on the headline.
7. **Validate on an overlap**, if one exists. Here there's no month where a city ran both systems, so compare like with like: the same city's tests per booking, cancellation rate and collected share before and after the switch, and look for a step change at April 2026.

**The trap:** comparing **totals across the system change without the rule** and attributing the difference to the business, for example "Chennai bookings rose in Q2 2026" when nova merely counts reschedules twice, or "Chennai cancellations doubled" when collections are just keyed to the old id. A second trap is applying the rule only halfway (dropping the originals but not re-pointing the downstream records that referenced them), which is what happened in this notebook.

## 2 · DO: collections, double scans, and the join that must not grow

#### 📘 Explanation: loading collections and removing double scans

- `coll_files = sorted(... "collections").glob("*.csv.gz"))`: 108 monthly files.
- The odd line `loaders.read_many(...) if False else pd.concat([...])`: the `if False` means the left side **never runs**. It's dead code left from an experiment (`read_many` only knows "legacy" and "nova" readers, so it couldn't read collections anyway). What actually runs is `pd.concat([pd.read_csv(p, dtype={"booking_id": "string"}) for p in coll_files])`, a sequential read of all collection files. You could simplify the line to just that.
- `pd.to_datetime(coll["collected_at"], errors="coerce")`: invalid timestamps become `NaT`.
- `coll.duplicated(subset=["booking_id","status","collected_at"], keep="first")` marks a row True if an **identical** (booking, status, timestamp) row appeared earlier. A phlebotomist scanning the same tube twice produces exactly that. `keep="first"` keeps the first occurrence.
- `coll_clean = coll[~dups]`, then two checks: the reconciliation, and **one row per booking** (`is_unique`).

In [4]:
coll_files = sorted((CONFIG["raw"] / "collections").glob("*.csv.gz"))
coll = loaders.read_many(coll_files, "legacy", workers=1) if False else pd.concat([pd.read_csv(p, dtype={"booking_id": "string"}) for p in coll_files], ignore_index=True)
coll["collected_at"] = pd.to_datetime(coll["collected_at"], errors="coerce")
n_in = len(coll)
dups = coll.duplicated(subset=["booking_id", "status", "collected_at"], keep="first")
coll_clean = coll[~dups].copy()
checks.reconciles("collections exact duplicates (double scans)", n_in, len(coll_clean), int(dups.sum()))
checks.check("one collection row per booking", coll_clean["booking_id"].is_unique, f"{(~coll_clean['booking_id'].is_unique)} duplicates remain" if not coll_clean["booking_id"].is_unique else "")

PASS  collections exact duplicates (double scans): input = kept + rejected   552,492 = 549,598 + 2,894
PASS  one collection row per booking


True

#### 📘 Reading the output

- **PASS: 552,492 = 549,598 + 2,894**: 2,894 double scans removed (0.52%).
- **PASS: one collection row per booking.** After removing exact duplicates, no booking has two *different* collection rows (for example "no_show" and later "collected"). If it had, that would need a rule such as "keep the latest status". The check proves we don't need one.
- The trailing `True` is the return value of the last check.

#### 📘 Explanation: the join that must not grow

`bookings.merge(coll_clean[[...]], on="booking_id", how="left", validate="one_to_one", indicator=True)`:
- `how="left"`: keep **every booking**, matched or not, like SQL `LEFT JOIN`. Unmatched bookings get NaN collection columns.
- `validate="one_to_one"`: pandas **checks** that `booking_id` is unique on both sides, and raises `MergeError` otherwise. A fan-out is caught *before* it inflates anything.
- `indicator=True` adds a `_merge` column: `both`, `left_only` (booking with no collection) or `right_only` (impossible in a left join, so always 0).
- The `try/except MergeError` is a fallback: if validation fails, it de-duplicates collections by `booking_id` and joins again. It didn't trigger here.
- `checks.join_did_not_fan_out(...)` checks `len(joined) == len(bookings)`.

In [5]:
# LEFT join bookings to collections. The row count must not grow. validate= makes pandas raise rather than lie.
try:
    joined = bookings.merge(coll_clean[["booking_id", "status", "collected_at", "phlebotomist_id"]], on="booking_id", how="left", validate="one_to_one", indicator=True)
except pd.errors.MergeError as err:
    print("MergeError:", err)
    joined = bookings.merge(coll_clean.drop_duplicates("booking_id")[["booking_id", "status", "collected_at", "phlebotomist_id"]], on="booking_id", how="left", validate="one_to_one", indicator=True)
checks.join_did_not_fan_out("bookings LEFT JOIN collections", bookings, joined, "booking_id")
joined["_merge"].value_counts()

PASS  bookings LEFT JOIN collections: join kept the left row count   left 563,380, joined 563,380, rows on repeated keys 0


_merge
both          545111
left_only      18269
right_only         0
Name: count, dtype: int64

#### 📘 Reading the output

- **PASS: left 563,380, joined 563,380**, so the join didn't fan out.
- `both` **545,111**: bookings with a collection record (collected or no-show).
- `left_only` **18,269**: bookings with **no** collection record.
- `right_only` 0, as always for a left join. (Separately, about 4,500 collection rows have no booking. They belong to rejected bookings, dropped originals and malformed ids, and are worth an anti-join list.)

### SEE: what `left_only` means

A booking with no collection row is a cancellation (the phlebotomy app does not record cancelled bookings) or a booking in the future at export time. Decide which, and write it down. A row that is `left_only` and dated months ago is a question for the data team, and a caveat on your slide.

#### ✅ Answer: what does `left_only` mean here? "Decide which, and write it down"

I broke the 18,269 unmatched bookings down by month and system:

| Group | Unmatched | Share of its bookings | Interpretation |
|---|---|---|---|
| legacy, every month Jan 2025 – Jun 2026 | ~850–1,000 per month (about 16,100 total) | **3.0%** | A steady, flat rate. These are **cancellations**: the phlebotomy app doesn't record cancelled bookings. |
| nova reschedule records (Apr–Jun 2026) | **1,500** | 100% of reschedules | **Not cancellations.** The collection was logged against the *original* id, which we dropped in cell 5. 1,449 of the 1,500 originals have a collection row. |
| other nova, slots Apr–Jun 2026 | 623 | about 2.7% | Cancellations, at a similar rate to legacy |
| nova slots in July 2026 | 44 | 100% | **Future** at export time, so not yet collected |

**Decision to write down:**
1. Legacy and non-reschedule nova `left_only` → **cancelled** (keep the row, status "cancelled").
2. Nova reschedules → **re-point** to the original's collection (the fix below).
3. Slots after the export date → "future", and outside the period anyway.

**Caveat for the slide:** the jump in "cancelled_or_missing" in April–June 2026 (from ~920 to ~1,420 a month, concentrated in Chennai and Jaipur) is an **artefact of the system change**, not a business change.

#### 📘 Explanation: labelling unmatched rows and flagging clock skew

- `joined["status"].fillna("cancelled_or_missing")`: unmatched bookings get an explicit status instead of NaN, so they're visible in every groupby.
- `skew = joined["collected_at"] < joined["slot_at"] - pd.Timedelta(days=1)`: a sample "collected" more than a day **before** its slot is physically implausible, most likely a device clock error. (Comparisons with NaT give False, so unmatched rows aren't flagged.)
- `joined.loc[skew, "collected_at"] = pd.NaT`: **keep the row, blank the bad timestamp**. The booking still counts. Only the unreliable time is removed.
- `joined["flag_clock_skew"] = skew` keeps a flag column, so the decision can be audited and reversed.
- `bookings = joined.drop(columns="_merge")`: the indicator has done its job.

In [6]:
joined["status"] = joined["status"].fillna("cancelled_or_missing")
skew = joined["collected_at"] < joined["slot_at"] - pd.Timedelta(days=1)
print("collected more than a day before the slot (clock skew?):", int(skew.sum()))
joined.loc[skew, "collected_at"] = pd.NaT       # keep the row, blank the timestamp, flag it
joined["flag_clock_skew"] = skew
bookings = joined.drop(columns="_merge")

collected more than a day before the slot (clock skew?): 936


#### 📘 Reading the output
**936** bookings had a collection timestamp more than a day before the slot. They're kept and flagged, with the timestamp blanked. (The tolerance of 1 day is a choice. Write it in the decisions log with the count.)

---
#### 🛠️ Suggested fix for the reschedule problem (not applied, so the original outputs stay valid)

Join collections on the id the phlebotomy app used, which is the original id for reschedules:
```python
# instead of on="booking_id" in cell 10:
bookings["coll_key"] = bookings["rescheduled_from"].fillna(bookings["booking_id"])
joined = bookings.merge(
    coll_clean[["booking_id", "status", "collected_at", "phlebotomist_id"]].rename(columns={"booking_id": "coll_key"}),
    on="coll_key", how="left", validate="one_to_one", indicator=True)
```
**Tested on the saved clean data:** no `coll_key` repeats, `validate="one_to_one"` passes, and the row count stays 563,380. `left_only` falls from 18,269 to **16,820**. Of the 1,500 rescheduled nova visits, **1,239 become collected, 210 no-show, and 51 stay unmatched** (genuine cancellations). Any headline that counts only collected bookings rises in H1 2026 for Chennai and Jaipur. Log it as a decision with that row count.

> **Interview angle.** Explain `how='left'`, `validate='one_to_one'` and `indicator=True` to someone who has only used SQL.
>
> *What a good answer covers:* LEFT keeps every left row whether or not it matches, like SQL LEFT JOIN. validate asserts the key's cardinality on each side and raises if it is violated, which SQL does not do for you. indicator adds a column saying which side each row came from, which is the SQL anti-join check (WHERE right.key IS NULL) made explicit.

#### ✅ Answer: explain `how='left'`, `validate='one_to_one'` and `indicator=True` to someone who only knows SQL

| pandas | SQL equivalent | What it does |
|---|---|---|
| `bookings.merge(coll, on="booking_id", how="left")` | `SELECT ... FROM bookings b LEFT JOIN coll c ON c.booking_id = b.booking_id` | Keeps **every left row**. Where there's no match, the right-hand columns are NULL (NaN). |
| `validate="one_to_one"` | *No direct equivalent.* SQL just returns more rows. The closest is a `UNIQUE` constraint, or running `SELECT booking_id, COUNT(*) ... GROUP BY 1 HAVING COUNT(*) > 1` on both tables first. | Pandas **asserts the cardinality** of the key on both sides and **raises** if it's violated. `"one_to_many"` requires the left key to be unique, and `"many_to_one"` the right key. This turns a silent double count into an error. |
| `indicator=True` | `CASE WHEN c.booking_id IS NULL THEN 'left_only' ELSE 'both' END AS _merge` | Adds a column telling you where each row came from. `left_only` rows are exactly the **anti-join** `WHERE c.booking_id IS NULL`: bookings without a collection. With `how="outer"` you also see `right_only`, the FULL OUTER JOIN orphans. |

Together they replace three separate SQL queries (the join, a duplicate-key check, an anti-join) with one line that checks itself.

## 3 · DO: patients, and the duplicate that looks different

#### 📘 Explanation: patient de-duplication

The patient master has one row per **patient_id**, but the same **person** can have several ids (registered twice, at two clinics, via app and phone). Counting ids overstates "people", which is the *people* branch of the growth tree in notebook 05.

- `pd.read_csv(..., dtype="string")`: everything as text, so phone numbers and dates aren't mangled (leading zeros, `+91`).
- `loaders.normalise_phone`: removes non-digits and keeps the **last 10**. `+91 98765 43210`, `09876543210` and `9876543210` all become `9876543210`.
- `name_norm`: upper-case, collapse repeated whitespace to one space, strip. `"rahul  fernandes "` becomes `"RAHUL FERNANDES"`.
- `pat.groupby(["phone_norm","name_norm","dob"]).ngroup()` gives every distinct (phone, name, dob) combination an integer, the `person_key`. Rows with the same key are treated as the same human. (No key column has missing values, so no row is dropped from grouping.)
- `pat.duplicated(subset=["person_key"], keep="first")`: every row after the first for each person is a duplicate.
- `bad_dob = pat["dob"].eq("1900-01-01")` counts the **placeholder** birth date (a default when the real one wasn't entered).
- `pat_clean = pat[~dup_mask]`: one row per person, so the first id seen is the surviving `person_id`.
- `id_map`: for **every** patient_id (duplicates included), the surviving id of its person. It's built by merging `pat` with `pat_clean` on `person_key`.
- `bookings.merge(id_map, on="patient_id", how="left", validate="many_to_one")`: each booking gets a `person_id`. Many bookings map to one person, and each patient_id appears once in `id_map`, which `validate` enforces.
- Final check: every booking found a person.

In [7]:
pat = pd.read_csv(CONFIG["raw"] / "patients" / "patients.csv.gz", dtype="string")
pat["phone_norm"] = loaders.normalise_phone(pat["phone"])
pat["name_norm"] = pat["patient_name"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
grp = pat.groupby(["phone_norm", "name_norm", "dob"]).ngroup()
pat["person_key"] = grp
dup_mask = pat.duplicated(subset=["person_key"], keep="first")
print(f"{len(pat):,} patient rows, {pat['person_key'].nunique():,} distinct people by phone+name+dob, {int(dup_mask.sum()):,} duplicate rows")
bad_dob = pat["dob"].eq("1900-01-01")
print("placeholder dob rows:", int(bad_dob.sum()))
pat_clean = pat[~dup_mask].copy()
checks.reconciles("patients de-duplicated", len(pat), len(pat_clean), int(dup_mask.sum()))
# map every duplicate id to its surviving id so bookings can be re-keyed to people, not to ids
id_map = pat.merge(pat_clean[["person_key", "patient_id"]].rename(columns={"patient_id": "person_id"}), on="person_key")[["patient_id", "person_id"]]
bookings = bookings.merge(id_map, on="patient_id", how="left", validate="many_to_one")
checks.check("every booking patient id maps to a person", bool(bookings["person_id"].notna().all()), int(bookings["person_id"].isna().sum()))

309,217 patient rows, 303,153 distinct people by phone+name+dob, 6,064 duplicate rows
placeholder dob rows: 1492
PASS  patients de-duplicated: input = kept + rejected   309,217 = 303,153 + 6,064
PASS  every booking patient id maps to a person   0


True

#### 📘 Reading the output

- **309,217 patient rows → 303,153 distinct people; 6,064 duplicate rows (2.0%).**
- **1,492 rows with the placeholder dob 1900-01-01.** Placeholder dobs make the rule weaker in both directions. The same person entered once with a real dob and once with 1900-01-01 **won't** merge. Two different people sharing a phone (family members) with the same name and both on the placeholder **would** merge. The volume is small, but log it.
- **PASS** reconciliation, and **PASS** every booking's patient id maps to a person (0 unmapped).

### The decision here is not technical

Merging two patient rows says two records are one human. For a growth analysis the cost of being wrong is a slightly wrong frequency figure. For a results system it would be a wrong report sent to the wrong person. Same rule, different stakes. Write the stakes in the decisions log beside the rule.

#### 📘 Explanation: "the decision here is not technical"

The code is the same whatever the use. What changes is the **cost of being wrong**:
- In a **growth analysis**, merging two different people (false positive) or missing a duplicate (false negative) shifts the "people" count by a fraction of a percent. Low stakes.
- In a **results system**, a false merge attaches person A's blood results to person B's record, and a report goes to the wrong patient. Very high stakes, so you'd require stricter evidence (a verified phone *and* an exact dob that isn't the placeholder) and human review.

Write **both** in the decisions log: the rule (phone + normalised name + dob), and the stakes that justify using a rule this loose here.

## 4 · DO: invoices and payments, two id families

#### 📘 Explanation: invoices and payments, two id families and three unit conversions

**Invoices:**
- `pd.concat([pd.read_csv(p, dtype="string") ...])`: all quarterly invoice files, read as text.
- `inv["amount"].str.replace(",", "", regex=False).astype(float)`: `"1,499.00"` becomes `"1499.00"`, then `1499.0`. `regex=False` because "," is literal text.
- `pd.to_datetime(inv["invoice_date"])` parses the dates.

**Payments** (the gateway export, `gateway_*` files only):
- `pay["amount_rs"] = pay["amount_paise"] / 100` converts **paise to rupees**.
- `pd.to_datetime(pay["created_at_epoch"], unit="s")`: epoch seconds become a **UTC** timestamp. `+ pd.Timedelta(hours=5, minutes=30)` shifts it to **IST**, so payment times compare correctly with invoice dates and the quarter cutoff (otherwise a payment at 02:00 IST on 1 July would look like 30 June UTC).
- `pay["invoice_id"] = loaders.normalise_invoice_ref(pay["notes_invoice_ref"])` gives the canonical `INV-YYYY-NNNNNN` (see notebook 01). The original reference column is kept.

In [8]:
inv = pd.concat([pd.read_csv(p, dtype="string") for p in sorted((CONFIG["raw"] / "invoices").glob("*.csv.gz"))], ignore_index=True)
inv["amount_rs"] = inv["amount"].str.replace(",", "", regex=False).astype(float)
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"])
pay = pd.concat([pd.read_csv(p, dtype={"notes_invoice_ref": "string", "settlement_id": "string"}) for p in sorted((CONFIG["raw"] / "payments").glob("gateway_*.csv.gz"))], ignore_index=True)
pay["amount_rs"] = pay["amount_paise"] / 100
pay["created_at"] = pd.to_datetime(pay["created_at_epoch"], unit="s") + pd.Timedelta(hours=5, minutes=30)
pay["invoice_id"] = loaders.normalise_invoice_ref(pay["notes_invoice_ref"])
print(f"invoices {len(inv):,} for Rs {inv['amount_rs'].sum():,.0f};  payment rows {len(pay):,} ({(pay.status=='captured').sum():,} captured, {(pay.status=='failed').sum():,} failed)")
checks.check("invoice ids unique", inv["invoice_id"].is_unique)

invoices 470,776 for Rs 737,020,069;  payment rows 473,507 (455,469 captured, 18,038 failed)
PASS  invoice ids unique


True

#### 📘 Reading the output

- **470,776 invoices for ₹737,020,069** (about ₹73.7 crore) over Jan 2025 – Jun 2026.
- **473,507 payment rows: 455,469 captured, 18,038 failed.** Failed rows are retries or declined attempts. They must never be counted as money received.
- **PASS: invoice ids unique**, so invoices can be the "one" side of the join.

#### 📘 Explanation: matching invoices to captured payments

- `captured = pay[pay["status"] == "captured"]`: only money that actually moved. Filtering **before** the join removes failed retries.
- `inv.merge(captured[...], on="invoice_id", how="outer", indicator=True, validate="one_to_many")`:
  - `how="outer"` keeps unmatched rows from **both** sides: invoices never paid, *and* payments with no invoice.
  - `validate="one_to_many"`: invoice ids must be unique (the left side). Payments may repeat.
- `m[m["_merge"]=="both"].groupby("invoice_id").size()` counts rows per matched invoice. More than 1 means **several captured payments for one invoice**.

In [9]:
captured = pay[pay["status"] == "captured"]
m = inv.merge(captured[["invoice_id", "payment_id", "amount_rs", "created_at"]].rename(columns={"amount_rs": "paid_rs"}), on="invoice_id", how="outer", indicator=True, validate="one_to_many")
print(m["_merge"].value_counts())
multi = m[m["_merge"] == "both"].groupby("invoice_id").size()
print("invoices with more than one captured payment:", int((multi > 1).sum()))

_merge
both          450881
left_only      21820
right_only      4588
Name: count, dtype: int64
invoices with more than one captured payment: 1925


#### 📘 Reading the output

| `_merge` | Rows | Meaning |
|---|---|---|
| both | 450,881 | invoice–payment pairs (an invoice with 2 payments appears twice) |
| left_only | **21,820** | invoices with **no** captured payment: unpaid, or paid in a way the gateway doesn't see |
| right_only | **4,588** | captured payments whose reference points to **no invoice** (₹7.27 M). Their refs are in an `INV-2025-9xxxxx` range that billing never issued. |

**1,925 invoices have more than one captured payment**: double charges, split payments or partial payments. Each one is a row-count *growth* in the join, and each one would double-count revenue if you summed invoice amounts after the join.

> **Interview angle.** Your LEFT join of invoices to payments grew the row count. Name three possible causes and the check that separates them.
>
> *What a good answer covers:* Retries (same invoice, a failed and a captured row): filter by status before joining. Genuine double charges (two captured rows): group by invoice and count captured. A many-to-many on a non-unique key such as amount and date: check key uniqueness on the right side with validate='one_to_many' or a groupby count before the join.

#### ✅ Answer: your LEFT join of invoices to payments grew the row count. Three possible causes, and the check that separates them

| Cause | What it looks like | The check that identifies it | The fix |
|---|---|---|---|
| **1. Retries** | the same invoice has a *failed* row and then a *captured* row | `pay.groupby("invoice_id")["status"].value_counts().unstack()`: invoices with ≥1 failed and exactly 1 captured | **filter `status == "captured"` before joining** (done above). 18,038 failed rows are excluded. |
| **2. Genuine multiple captures** (double charge, split or partial payment) | two or more *captured* rows for one invoice | `captured.groupby("invoice_id").size() > 1` → **1,925 invoices**. Compare the summed paid amount with the invoice amount: sum ≈ 2× invoice means a double charge (refund due), sum ≈ invoice means a split payment. | Aggregate payments to **one row per invoice** (`sum`, `count`, `min(created_at)`) *before* joining, as notebook 05's bridge does. Flag overpayments for finance. |
| **3. A many-to-many join on a non-unique key** (for example joining on amount + date, or on a key that repeats on *both* sides) | the row count explodes far beyond the number of payments | check uniqueness of the key on each side **before** the join: `inv["invoice_id"].is_unique`, and use `validate="one_to_many"` so pandas raises if the left side repeats | join on a unique key, or de-duplicate the side that should be unique |

The order to diagnose: run `validate=` (rules out 3), then count captured rows per invoice (separates 2), then count failed rows (separates 1).

## 5 · SUM: write the clean tables and log every rule

#### 📘 Explanation: write the five clean tables

Each table is saved gzip-compressed to `data/clean/`. The headline is printed one last time ("after cleaning"), and `checks.summary()` lists every check of this notebook. The `logs.decision(...)` and `logs.reconciliation(...)` rows are still to be written into `docs/decisions-log.md`. The ✅ answer below gives the full list.

In [10]:
bookings.to_csv(CONFIG["clean"] / "bookings_clean.csv.gz", index=False, compression="gzip")
coll_clean.to_csv(CONFIG["clean"] / "collections_clean.csv.gz", index=False, compression="gzip")
pat_clean.to_csv(CONFIG["clean"] / "patients_clean.csv.gz", index=False, compression="gzip")
inv.to_csv(CONFIG["clean"] / "invoices_clean.csv.gz", index=False, compression="gzip")
pay.to_csv(CONFIG["clean"] / "payments_clean.csv.gz", index=False, compression="gzip")
print("headline after cleaning:", HEADLINE(bookings))
checks.summary()

headline after cleaning: 563,380 bookings, 1,303,719 tests
7 of 7 checks passed


,check,passed,detail
0,drop superseded nova records: input = kept + r...,True,"564,879 = 563,380 + 1,499"
1,collections exact duplicates (double scans): i...,True,"552,492 = 549,598 + 2,894"
2,one collection row per booking,True,
3,bookings LEFT JOIN collections: join kept the ...,True,"left 563,380, joined 563,380, rows on repeated..."
4,patients de-duplicated: input = kept + rejected,True,"309,217 = 303,153 + 6,064"
5,every booking patient id maps to a person,True,0
6,invoice ids unique,True,


#### 📘 Reading the output
**563,380 bookings, 1,303,719 tests** after cleaning. **7 of 7 checks passed.** Every step reconciles and no join fanned out.

## Before you move on

- List every rule you applied, in order, with the headline number before and after each. That list is `docs/decisions-log.md`; if it is not there yet, stop and write it.
- Which rule moved the headline most? That rule is on slide 5.
- Which rule would the finance controller challenge first?

#### ✅ Answers: "Before you move on"

**1. Every rule applied, in order, with the headline before and after**

Two headlines are tracked: the notebook's own (all bookings and tests in the pack), and the one that matters for the claim, **test growth H1 2026 vs H1 2025** (full-day periods: 1 Jan – 30 Jun inclusive).

| # | Rule (notebook) | Rows affected | Bookings / tests after | H1 growth after |
|---|---|---|---|---|
| 0 | raw as received (02) | n/a | 567,986 bookings / 1,315,183 booking-test rows | **+5.78%** (436,302 → 461,523 rows) |
| 1 | drop repeated header lines (01/02) | 21 lines (not data) | unchanged | unchanged |
| 2 | strip whitespace from ids, map `Bangalore` → `Bengaluru` (02) | ids / city labels only | unchanged | unchanged (changes per-city splits) |
| 3 | validator rejects (02) | **3,107 bookings** (1,027 slot-before-creation, 793 bad ids, 1,287 empty test codes) | 564,879 / 1,307,260 | **+5.84%** |
| 4 | drop superseded nova originals (03) | **1,499 bookings / 3,541 tests**, all H1 2026 | **563,380 / 1,303,719** | **+5.02%** (433,611 → 455,383) |
| 5 | remove collection double scans (03) | 2,894 collection rows | unchanged | unchanged |
| 6 | unmatched bookings → `cancelled_or_missing` (03) | 18,269 bookings labelled | unchanged | unchanged (changes any *collected-only* count) |
| 7 | blank clock-skew timestamps (03) | 936 timestamps | unchanged | unchanged |
| 8 | patient de-duplication (03) | 6,064 ids merged into others | unchanged bookings; people 303,153 | unchanged tests (changes the *people* branch) |
| 9 | parse invoice amount, paise → rupees, epoch → IST, normalise payment refs (03) | all invoice and payment rows | n/a | n/a (money tables) |

**2. Which rule moved the headline most?**

**Rule 4, dropping the superseded nova originals.** It removes 3,541 tests from H1 2026 only (Chennai and Jaipur, April–June), taking growth from +5.84% to +5.02%, **about −0.8 points**. The validator rejects (rule 3) remove more bookings (3,107), but they're spread evenly across both years, so they move growth by only about +0.06 points. That's slide 5: *"our growth figure depends on treating a nova reschedule as one visit. Counted as two, growth would be +5.8%."*

**3. Which rule would the finance controller challenge first?**

**The invoice–payment matching (rule 9)**, because it decides what money counts as collected:
- **payment-reference normalisation** rewrites about 30% of references, so they'll want the rule, the row count and a sample;
- the **4,588 captured payments (₹7.27 M) that match no invoice**: is that unallocated cash, a billing range we're missing, or test data?
- the **1,925 invoices with multiple captured payments**: double charges owed back to patients, or split payments?

On the bookings side they'd also question **rule 3** (rejecting 3,107 bookings that were probably real visits, some of them invoiced) and the **reschedule–collection mismatch** (1,449 visits with a collection record shown as cancelled).